# 01 · Duplicate check: BePLi v2 vs BeachLitter v2022

BePLi v2 and BeachLitter v2022 come from the same monitoring programme in Yamagata, so some photos appear in both. This notebook finds them with a perceptual hash and confirms each match by pixel difference. Duplicates are removed from BePLi so that no image is used twice across the Japanese training, validation and test sets.

Output: the list of BePLi images that do not appear in v2022.

## 1. Packages

In [ ]:
!pip install -q imagehash

## 2. Configuration

| Parameter | Value | Meaning |
|---|---|---|
| `HASH_SIZE` | 16 | 256-bit perceptual hash |
| `HAMMING_MAX` | 12 | maximum hash distance for a candidate pair |
| `COMPARE_SIZE` | 256 × 256 | grayscale size for the pixel check |
| `MAD_MAX` | 4.0 | maximum mean absolute pixel difference (0–255) for two images to count as the same |

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import imagehash
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

BEPLI_DIR = Path("/kaggle/input/datasets/rafiadabhi/databepliv2/original_images")
V2022_DIR = Path("/kaggle/input/datasets/rafiadabhi/the-beachlitter-dataset-v2022/beachlitter_dataset_2022/beachlitter/images")
OUT_DIR = Path("/kaggle/working")

HASH_SIZE = 16
HAMMING_MAX = 12
COMPARE_SIZE = (256, 256)
MAD_MAX = 4.0

## 3. Functions

In [ ]:
POPCOUNT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)


def list_images(folder: Path, ext: str) -> list[Path]:
    return sorted(folder.glob(f"*{ext}"))


def phash_bits(path: Path) -> np.ndarray:
    with Image.open(path) as img:
        h = imagehash.phash(img.convert("RGB"), hash_size=HASH_SIZE)
    return np.packbits(h.hash.flatten())


def compute_hashes(paths: list[Path], desc: str) -> np.ndarray:
    with ProcessPoolExecutor() as pool:
        hashes = list(tqdm(pool.map(phash_bits, paths, chunksize=32), total=len(paths), desc=desc))
    return np.stack(hashes)


def find_candidates(hash_a: np.ndarray, hash_b: np.ndarray, max_dist: int, block: int = 256) -> pd.DataFrame:
    rows = []
    for start in range(0, len(hash_a), block):
        xor = hash_a[start:start + block, None, :] ^ hash_b[None, :, :]
        dist = POPCOUNT[xor].sum(axis=-1, dtype=np.int32)
        for i, j in zip(*np.nonzero(dist <= max_dist)):
            rows.append((start + i, j, int(dist[i, j])))
    return pd.DataFrame(rows, columns=["idx_bepli", "idx_v2022", "hamming"])


def load_gray(path: Path) -> np.ndarray:
    with Image.open(path) as img:
        return np.asarray(img.convert("L").resize(COMPARE_SIZE, Image.BILINEAR), dtype=np.float32)


def image_size(path: Path) -> str:
    with Image.open(path) as img:
        return f"{img.width}x{img.height}"


def verify_pairs(cand: pd.DataFrame, bepli: list[Path], v2022: list[Path]) -> pd.DataFrame:
    records = []
    for row in tqdm(cand.itertuples(index=False), total=len(cand), desc="Pixel check"):
        pa, pb = bepli[row.idx_bepli], v2022[row.idx_v2022]
        mad = float(np.abs(load_gray(pa) - load_gray(pb)).mean())
        records.append({
            "file_bepli": pa.name,
            "file_v2022": pb.name,
            "size_bepli": image_size(pa),
            "size_v2022": image_size(pb),
            "hamming": row.hamming,
            "mad": round(mad, 2),
            "identical": mad <= MAD_MAX,
        })
    return pd.DataFrame(records).sort_values(["mad", "hamming"]).reset_index(drop=True)


def show_pairs(df: pd.DataFrame, title: str, n: int = 10) -> None:
    df = df.head(n)
    if df.empty:
        print(f"{title}: none.")
        return
    fig, axes = plt.subplots(len(df), 2, figsize=(10, 3.2 * len(df)), squeeze=False)
    for ax_row, row in zip(axes, df.itertuples(index=False)):
        for ax, folder, name in zip(ax_row, (BEPLI_DIR, V2022_DIR), (row.file_bepli, row.file_v2022)):
            with Image.open(folder / name) as img:
                img.thumbnail((512, 512))
                ax.imshow(img.convert("RGB"))
            ax.set_title(name, fontsize=9)
            ax.axis("off")
        ax_row[0].text(0, -0.08, f"hamming={row.hamming} | mad={row.mad}",
                       transform=ax_row[0].transAxes, fontsize=8, va="top")
    fig.suptitle(f"{title} (left: BePLi v2, right: v2022)", fontsize=12)
    fig.tight_layout()
    plt.show()

## 4. Hash both datasets

In [ ]:
bepli_paths = list_images(BEPLI_DIR, ".png")
v2022_paths = list_images(V2022_DIR, ".jpg")
print(f"BePLi v2: {len(bepli_paths)} images")
print(f"v2022:   {len(v2022_paths)} images")

bepli_hash = compute_hashes(bepli_paths, "Hashing BePLi")
v2022_hash = compute_hashes(v2022_paths, "Hashing v2022")

## 5. Candidates and pixel check

Every BePLi–v2022 pair within `HAMMING_MAX` is a candidate. A candidate is a duplicate when its mean absolute difference is at most `MAD_MAX`; the rest are listed for manual review.

In [ ]:
candidates = find_candidates(bepli_hash, v2022_hash, HAMMING_MAX)
print(f"Candidate pairs (Hamming <= {HAMMING_MAX}): {len(candidates)}")

pairs = verify_pairs(candidates, bepli_paths, v2022_paths) if len(candidates) else pd.DataFrame(
    columns=["file_bepli", "file_v2022", "size_bepli", "size_v2022", "hamming", "mad", "identical"])

duplicates = pairs[pairs["identical"]].reset_index(drop=True)
borderline = pairs[~pairs["identical"]].reset_index(drop=True)

print(f"Duplicates (MAD <= {MAD_MAX}): {len(duplicates)} pairs, "
      f"{duplicates['file_bepli'].nunique()} BePLi images")
print(f"Similar but not identical (manual review): {len(borderline)} pairs")

## 6. Matched pairs
The first ten duplicates side by side (left: BePLi v2, right: v2022).

In [ ]:
display(duplicates)
show_pairs(duplicates, "Duplicates")
show_pairs(borderline, "Manual review")

## 7. Output

| File | Content |
|---|---|
| `duplicates_bepli_v2022.csv` | duplicate pairs |
| `usable_bepli.csv` | BePLi images not found in v2022 |
| `all_candidates.csv` | every candidate pair, including those needing manual review |

In [ ]:
dup_names = set(duplicates["file_bepli"])
usable = pd.DataFrame({"file": [p.name for p in bepli_paths if p.name not in dup_names]})

pairs.to_csv(OUT_DIR / "all_candidates.csv", index=False)
duplicates.to_csv(OUT_DIR / "duplicates_bepli_v2022.csv", index=False)
usable.to_csv(OUT_DIR / "usable_bepli.csv", index=False)

print(f"Usable BePLi images: {len(usable)} of {len(bepli_paths)}")